# BirdWeather PUC – erster Überblick (Deutschland, 22.10.2026)

Daten: Ausgabe von `birdweather_puc_germany.py` (Stationen, Environment-Sensoren, Light-Sensor, Detections).
Ziel: nur ein Eindruck, kein fertiges Auswertungsnotebook.

In [ ]:
from pathlib import Path
import pandas as pd
import plotly.express as px

DAY = "2026-10-22"
BASE = Path("birdweather_out") #/ DAY   # Ordner, den das Abruf-Skript erzeugt hat
TZ = "Europe/Berlin"                   # alle Stationen liegen in Deutschland
pd.set_option("display.max_columns", 30)

## 1. Laden

In [ ]:
def load(name, time_col="timestamp"):
    path = BASE / name
    if not path.exists():
        print(f"{name}: nicht gefunden")
        return pd.DataFrame(columns=["station_id", time_col, "local_time"])
    df = pd.read_csv(path, dtype={"station_id": str, "id": str})
    if time_col in df:
        df[time_col] = pd.to_datetime(df[time_col], utc=True)   # gemischte Offsets -> UTC
        df["local_time"] = df[time_col].dt.tz_convert(TZ)
    return df

stations = pd.read_csv(BASE / "stations.csv", dtype={"id": str})
env = load("sensor_environment.csv")
light = load("sensor_light.csv")
det = load("detections.csv")
print({"Stationen": len(stations), "env": len(env), "light": len(light), "detections": len(det)})

## 2. Welcher Zeitraum steckt in den Daten?

Das Abruf-Skript schneidet jeden Datentyp bereits auf den Zieltag (in der Stationszeit) zu.
Ein weiteres Datums-Filtern ist also nicht nötig. Diese Zelle prüft es nur.

In [ ]:
for name, df in [("environment", env), ("light", light), ("detections", det)]:
    if len(df):
        print(f"{name:12s} {df['local_time'].min()}  ->  {df['local_time'].max()}")
        print(f"{'':12s} Kalendertage: {sorted(df['local_time'].dt.date.unique())}")
    else:
        print(f"{name:12s} keine Daten")

## 3. Stationen mit Daten

In [ ]:
def n_per_station(df):
    return df.groupby("station_id").size() if len(df) else pd.Series(dtype="int64")

counts = pd.concat({"env": n_per_station(env),
                    "light": n_per_station(light),
                    "det": n_per_station(det)}, axis=1).fillna(0).astype(int)
counts.index.name = "station_id"
counts = counts.join(stations.set_index("id")[["name", "state", "location"]], how="left")

REQUIRE_ALL = False   # True: nur Stationen mit env UND light UND Detections
has = counts[["env", "light", "det"]] > 0
active = counts[has.all(axis=1) if REQUIRE_ALL else has.any(axis=1)]

print(f"{len(active)} von {len(stations)} Stationen haben Daten "
      f"({'alle drei Typen' if REQUIRE_ALL else 'mindestens ein Typ'})")
active.sort_values("env", ascending=False)

In [ ]:
# Alle weiteren Auswertungen nur mit diesen Stationen
env = env[env["station_id"].isin(active.index)]
light = light[light["station_id"].isin(active.index)]
det = det[det["station_id"].isin(active.index)]

## 4. Environment-Sensoren

Die Einheiten stehen nicht in der API-Doku (Temperatur vermutlich °C, Druck hPa) und sollten geprüft werden.

In [ ]:
env.select_dtypes("number").describe().T

In [ ]:
for col in ["temperature", "humidity", "barometricPressure", "voc"]:
    if col in env and env[col].notna().any():
        fig = px.line(env.sort_values("local_time"), x="local_time", y=col,
                      color="station_id", title=col)
        fig.show()

## 5. Light-Sensor

In [ ]:
if len(light):
    print(light.select_dtypes("number").describe().T[["count", "mean", "min", "max"]])
    for col in ["clear", "nir"]:
        if col in light and light[col].notna().any():
            px.line(light.sort_values("local_time"), x="local_time", y=col,
                    color="station_id", title=f"Light: {col}").show()
else:
    print("Keine Light-Daten")

## 6. Detections

In [ ]:
if len(det):
    per_station = det.groupby("station_id").size().sort_values(ascending=False).reset_index(name="n")
    px.bar(per_station, x="station_id", y="n", title="Detections pro Station").show()

    top = det.groupby("commonName").size().nlargest(15).sort_values().reset_index(name="n")
    px.bar(top, x="n", y="commonName", orientation="h", title="Häufigste Arten").show()

    per_hour = det.groupby(det["local_time"].dt.hour).size().reset_index(name="n")
    per_hour.columns = ["hour", "n"]
    px.bar(per_hour, x="hour", y="n", title="Detections nach Stunde (Stationszeit)").show()

    px.histogram(det, x="confidence", nbins=40, title="Verteilung der Konfidenz").show()
else:
    print("Keine Detections")

## 7. Karte

In [ ]:
m = (active.join(stations.set_index("id")[["lat", "lon"]])
           .dropna(subset=["lat", "lon"]).reset_index())
m["size"] = m["det"].clip(lower=1)
args = dict(lat="lat", lon="lon", color="det", size="size", hover_name="name",
            hover_data=["station_id", "env", "light", "det"], zoom=5,
            title="Stationen (Farbe/Größe = Anzahl Detections)")
try:
    fig = px.scatter_map(m, **args)
except AttributeError:   # ältere plotly-Versionen
    fig = px.scatter_mapbox(m, mapbox_style="open-street-map", **args)
fig.show()

## Mögliche nächste Schritte
- Einheiten und Plausibilität der Sensorwerte prüfen
- Detections pro Stunde mit Temperatur oder Licht vergleichen
- Weitere Tage abrufen und zusammenführen